# Process DDL or DML Updates
This this to update DDL/DML to update existing fact tables with new variables or create new fact tables.

In [0]:
# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

import json

# dbutils.widgets.text() is a built-in Databricks function that creates a free-form text input box at the top of a notebook.
#
# Use case is a run parameter — select it via the widget (or the job
# parameter when scheduled) instead of editing this cell for each use case.
dbutils.widgets.text("use_case_name", "bmi_statistics_by_wave_race_gender_age_cohort", "Use case name")  
dbutils.widgets.text("codebook_version", "1992-2022v1", "Codebook version")

# Get the values
use_case_name = dbutils.widgets.get("use_case_name")
codebook_version = dbutils.widgets.get("codebook_version")

# Set the path to the data dictionary notebook
data_dict_path = "../HRS_Silver_Data_Model_Specifications/dm_001_bmi_statistics_by_wave_race_gender_age_cohort"

# Set the path to 5.1.2 Approved Variable Processing
var_processing_path = "../meta_data_foundation/Step 1.5.2 Approved Variable Processing.ipynb"

# Set path to 5.1.3 Approved Cohort Coverage
cohort_processing_path = "../meta_data_foundation/Step 1.5.3 Approved Cohort Coverage.ipynb"

# Set the arguments
args = {"use_case_name": use_case_name, "codebook_version": codebook_version}

# -----------------------------------------------------------------------------------------
# 2. Processing
# Run the notebooks to process Silver table DDL/DML.
# Each called nootbook may take arguements and return a string from dbutils.notebook.exit()
# -----------------------------------------------------------------------------------------

#-----------------------------------------------------------------------
# 2.1 Run the target data dictionary notebook.
#-----------------------------------------------------------------------
try:
    # Attempt to run the child notebook
    #output = dbutils.notebook.run(data_dict_path, timeout_seconds=600)
    approved_dictionary = dbutils.notebook.run(data_dict_path, timeout_seconds=600)
    #combined_args = args | {"approved_dictionary": json.dumps(approved_dictionary)}
    combined_args = args | {"approved_dictionary": approved_dictionary}
    
    print(f"Success! Notebook returned: {output}")

except Exception as e:
    # Handle the failure (log it, send an alert, or set a fallback value)
    print(f"Failed to run notebook at '{data_dict_path}'.")
    print(f"Error details: {e}")
    raise

#-----------------------------------------------------------------------
# 2.2 Run the Step 1.5.2 Approved Variable Processing notebook
#-----------------------------------------------------------------------
try:
    # Attempt to run the child notebook
    output = dbutils.notebook.run(var_processing_path, timeout_seconds=600, arguments=combined_args)
    print(f"Success! Notebook 1.5.2 returned: {output}")

except Exception as e:
    # Handle the failure (log it, send an alert, or set a fallback value)
    print(f"Failed to run notebook 1.5.2 at '{var_processing_path}'.")
    print(f"Error details: {e}")
    raise

#-----------------------------------------------------------------------
# 2.3 Run the Step 1.5.3 Approved Cohort Coverage notebook
#-----------------------------------------------------------------------
try:
    # Attempt to run the child notebook
    output = dbutils.notebook.run(cohort_processing_path, timeout_seconds=600, arguments=args)
    print(f"Success! Notebook 1.5.3  returned: {output}")

except Exception as e:
    # Handle the failure (log it, send an alert, or set a fallback value)
    print(f"Failed to run notebook 1.5.3 at '{cohort_processing_path}'.")
    print(f"Error details: {e}")
    raise
